# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

In [2]:
import pandas as pd
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())

1 1.3333333333333333


In [3]:
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())

2


In [4]:
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))

3


In [5]:
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())

1005


In [6]:
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

[9, 10]


Ваши ответы и объяснения:

1. через изна он проверить есть ли там нан если да то он будет тру и возвращае один а на втором филлна это можна и сказать что заменить нал на 0 и просто 1+0+3/3 возвращаеть мин
2. через стрип удаляем пробелы потом всех буквы на листе через ловер кышы буквы сделаем потом через нонюник вернем количества уникальных
3. у нас дубликейт просто можна сказать удалить дубликаты но никак не может повлиять на исходную лист из за чего когда вы вызываем лен он просто исходной листтын длинасын кайтарады
4. ту нумерикал возвращает нам разные типы либо флоут или инт стр реплейс удалить у нас пробелы и потом просто суммирует через сам
5. через дейттаим у вы возвращаем стринг как дата после монз то лист вытащить из дата месяцы и просто возврашает их как лист

In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [8]:
df = pd.read_csv("orders.csv")
print("Бастапқы қатарлар:", len(df))


Бастапқы қатарлар: 5463


In [16]:
df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(len(df))

5463


In [17]:
df = df.drop_duplicates()
print(len(df))

df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
print(len(df))

5283
5283


In [18]:
df = df[df["customer_id"] != "TEST"]
print(len(df))

5280


In [20]:
print("число amount NaN :", df["amount"].isna().sum())
print("число city NaN :", df["city"].isna().sum())
print("order_id :", df["order_id"].is_unique)

число amount NaN : 0
число city NaN : 0
order_id : True


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [21]:
paid = df["status"] == "оплачен"
sep = df["month"] == 9
octo = df["month"] == 10

s, o = df.loc[paid & sep, "amount"].sum(), df.loc[paid & octo, "amount"].sum()
print(s, o, f"{o / s - 1:.1%}")

35243410.0 34419520.0 -2.3%


In [23]:
al = df["city"] == "Алматы"
s, o = df.loc[paid & sep & al, "amount"].sum(), df.loc[paid & octo & al, "amount"].sum()
print("Алматы:", s, o, f"{o / s - 1:.1%}")

ast = df["city"] == "Астана"
s, o = df.loc[paid & sep & ast, "amount"].sum(), df.loc[paid & octo & ast, "amount"].sum()
print("Астана:", s, o, f"{o / s - 1:.1%}")

sh = df["city"] == "Шымкент"
s, o = df.loc[paid & sep & sh, "amount"].sum(), df.loc[paid & octo & sh, "amount"].sum()
print("Шымкент:", s, o, f"{o / s - 1:.1%}")

kar = df["city"] == "Караганда"
s, o = df.loc[paid & sep & kar, "amount"].sum(), df.loc[paid & octo & kar, "amount"].sum()
print("Караганда:", s, o, f"{o / s - 1:.1%}")

akt = df["city"] == "Актобе"
s, o = df.loc[paid & sep & akt, "amount"].sum(), df.loc[paid & octo & akt, "amount"].sum()
print("Актобе:", s, o, f"{o / s - 1:.1%}")

aty = df["city"] == "Атырау"
s, o = df.loc[paid & sep & aty, "amount"].sum(), df.loc[paid & octo & aty, "amount"].sum()
print("Атырау:", s, o, f"{o / s - 1:.1%}")

Алматы: 12683170.0 12975290.0 2.3%
Астана: 8073960.0 7760480.0 -3.9%
Шымкент: 3873110.0 3517540.0 -9.2%
Караганда: 3648520.0 3371680.0 -7.6%
Актобе: 2576960.0 2540420.0 -1.4%
Атырау: 923990.0 969860.0 5.0%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [24]:
df_raw = pd.read_csv("orders.csv")
df_raw["amount"] = pd.to_numeric(df_raw["amount"].str.replace(" ", ""))


paid = df_raw["status"] == "оплачен"
sep = df_raw["order_date"] < "2026-10"
octo = ~sep
s0, o0 = df_raw.loc[paid & sep, "amount"].sum(), df_raw.loc[paid & octo, "amount"].sum()
print(f" {len(df_raw)}, {o0 / s0 - 1:.1%}")

 5463, -22.7%


In [26]:
df1 = df_raw[df_raw["customer_id"] != "TEST"]
paid1 = df1["status"] == "оплачен"
sep1 = df1["order_date"] < "2026-10"
octo1 = ~sep1
s1, o1 = df1.loc[paid1 & sep1, "amount"].sum(), df1.loc[paid1 & octo1, "amount"].sum()
print(f" {len(df1)} ( {len(df_raw) - len(df1)}), {o1 / s1 - 1:.1%}")

 5460 ( 3), -8.2%


In [27]:
df2 = df1.drop_duplicates()
paid2 = df2["status"] == "оплачен"
sep2 = df2["order_date"] < "2026-10"
octo2 = ~sep2
s2, o2 = df2.loc[paid2 & sep2, "amount"].sum(), df2.loc[paid2 & octo2, "amount"].sum()
print(f"{len(df2)} ( {len(df1) - len(df2)}), {o2 / s2 - 1:.1%}")

5280 ( 180), -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
# 4

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [44]:
df = pd.read_csv("orders.csv")
df=df.drop_duplicates()
df["city"] = df["city"].str.strip().str.lower().replace(" ", "")
print(df["city"].nunique(), len(df))

9 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8